# Práctica 7: PyTorch: autograd y entrenamiento de redes

**Fundamentos matemáticos del aprendizaje profundo** · Departamento de Matemática, FCEN-UBA

Material de referencia: capítulos 6 y 8 de las notas del curso. Convenciones: $y=\varphi(w\cdot x+b)$, $\sigma$ es la logística, $\eta$ la tasa de aprendizaje, $w_{ij}$ conecta la entrada $i$ con la salida $j$.

Cada práctica tiene una **Parte A** (ejercicios teóricos, para resolver en papel) y una **Parte B** (computacional). Las celdas marcadas `# TODO` son para completar.

## Parte A. Ejercicios teóricos

**A1.** Para $f(x)=\sin(x_1x_2)+x_2^2$ dibujar el grafo computacional y calcular $\nabla f$ por modo reverso, anotando los valores intermedios en $x=(1,2)$.

**A2.** Sea $u(x)=\sum_{j=1}^m c_j\,\mathrm{ReLU}(w_jx+b_j)$. Mostrar que $u$ es lineal a trozos con a lo sumo $m$ puntos de quiebre, ubicados en $-b_j/w_j$. Calcular $u'$ y $u''$ (como distribución) usando $\mathrm{ReLU}'=H$ y $\mathrm{ReLU}''=\delta$.

**A3.** Mostrar que $H'(x-x_0)=\delta(x-x_0)$ y $(x\,\mathrm{ReLU}(x))'=2\,\mathrm{ReLU}(x)$ en sentido de distribuciones.

**A4.** Sea $u(x)=\sum_j c_j\sigma(w_jx+b_j)$. Escribir el gradiente de $C=\tfrac1{2N}\sum_i(u(x_i)-y_i)^2$ respecto de $(c,w,b)$ y verificar que coincide con lo que calcularía autograd.

## Parte B. Computacional

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch
from torch import nn
torch.manual_seed(0)
plt.rcParams['figure.figsize']=(6,4)

### B1. Autograd
Verificar A1: calcular $\nabla f(1,2)$ con `torch.autograd` y a mano.

In [ ]:
x=torch.tensor([1.,2.],requires_grad=True)
f=torch.sin(x[0]*x[1])+x[1]**2
f.backward()
print(x.grad)
# TODO: comparar con la formula analitica

### B2. Una red simple con `nn.Module` y un loop de entrenamiento explícito
Ajustar $f(x)=\sin(3x)$ en $[-1,1]$ con una red `1-32-32-1` (tanh), costo MSE y descenso de gradiente, **escribiendo el loop a mano** (`loss.backward()`, actualización de parámetros con `torch.no_grad()`).

In [ ]:
N=200
x=torch.linspace(-1,1,N).unsqueeze(1); y=torch.sin(3*x)
red=nn.Sequential(nn.Linear(1,32),nn.Tanh(),nn.Linear(32,32),nn.Tanh(),nn.Linear(32,1))
eta=0.05; hist=[]
for ep in range(3000):
    loss=((red(x)-y)**2).mean()
    red.zero_grad(); loss.backward()
    with torch.no_grad():
        for p in red.parameters(): p-=eta*p.grad
    hist.append(loss.item())
print(hist[-1])
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
ax[0].semilogy(hist); ax[1].plot(x,y,label='f'); ax[1].plot(x,red(x).detach(),'--',label='red'); ax[1].legend(); plt.show()

**B2.1** Verificar que los gradientes de `autograd` coinciden con los de la `Red` de NumPy de la Práctica 5 (copiar los pesos de una a la otra; recordar que `nn.Linear` guarda $W$ de tamaño `salida×entrada`).

In [ ]:
# TODO

### B3. Optimizadores
Reentrenar con `torch.optim.SGD`, `SGD(momentum=0.9)` y `Adam` (misma semilla) y comparar las curvas de costo.

In [ ]:
# TODO

### B4. Redes ReLU y puntos de quiebre
Entrenar `Linear(1,m)–ReLU–Linear(m,1)` con $m=10$ sobre $f(x)=|x|^{1/2}$ en $[-1,1]$. Extraer los puntos de quiebre $-b_j/w_j$ de la capa oculta y marcarlos sobre el gráfico de la aproximación (A2).

In [ ]:
# TODO

### B5. Mini-batches con `DataLoader`
Repetir B2 usando `TensorDataset`, `DataLoader(batch_size=32, shuffle=True)` y `optim.Adam`. ¿Cuántas épocas hacen falta para alcanzar el mismo costo?

In [ ]:
# TODO